In [1]:
# ============================================================
# DATATHON 2026 — SKIPPED SESSION-LEVEL EDA FROM SCRATCH
# Không cần session_summary_sample.parquet có sẵn
#
# Mục tiêu:
# - Tự tạo sampled session summary từ raw fact_user_events
# - EDA sâu:
#   1. non_login_only làm gì
#   2. nonlogin_then_login_same_session làm gì
#   3. login_only làm gì
#   4. event/category nào overrepresented trong converted sessions
#   5. converted sessions xem city/category nào
#   6. converted sessions hoạt động giờ nào
#   7. dwell/session interaction intensity
#
# Event-level trong code này là SAMPLE SESSION, không phải full data.
# Vì phần này là session-level detail, nếu full data thì rất dễ đầy disk.
# ============================================================

import os
import glob
import gc
import sys
import shutil
import subprocess
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

try:
    import duckdb
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

try:
    import plotly.express as px
    import plotly.graph_objects as go
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "plotly"])
    import plotly.express as px
    import plotly.graph_objects as go

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# ============================================================
# 0. CONFIG
# ============================================================

BASE_PATH_2 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2"
EVENTS_PATH = os.path.join(BASE_PATH_2, "fact_user_events")

OUTPUT_DIR = "/kaggle/working/eda_skipped_session_detail_from_scratch"
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
CACHE_DIR = os.path.join(OUTPUT_DIR, "cache")

CLEAN_OUTPUT = True

if CLEAN_OUTPUT and os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

PARTIAL_SESSION_DIR = os.path.join(CACHE_DIR, "partial_session")
PARTIAL_EVENT_CAT_DIR = os.path.join(CACHE_DIR, "partial_session_event_category")
PARTIAL_UNIQUE_EVENT_CAT_DIR = os.path.join(CACHE_DIR, "partial_unique_session_event_category")
PARTIAL_CITY_CAT_DIR = os.path.join(CACHE_DIR, "partial_session_city_category")
PARTIAL_CONTACT_CITY_CAT_DIR = os.path.join(CACHE_DIR, "partial_session_contact_city_category")
PARTIAL_HOUR_DIR = os.path.join(CACHE_DIR, "partial_session_hour")
PARTIAL_DEVICE_SURFACE_DIR = os.path.join(CACHE_DIR, "partial_session_device_surface")

for d in [
    PARTIAL_SESSION_DIR,
    PARTIAL_EVENT_CAT_DIR,
    PARTIAL_UNIQUE_EVENT_CAT_DIR,
    PARTIAL_CITY_CAT_DIR,
    PARTIAL_CONTACT_CITY_CAT_DIR,
    PARTIAL_HOUR_DIR,
    PARTIAL_DEVICE_SURFACE_DIR,
]:
    os.makedirs(d, exist_ok=True)

EVENT_FILES = sorted(glob.glob(os.path.join(EVENTS_PATH, "*.parquet")))

BATCH_SIZE = 300_000
TOP_N = 30
DWELL_CAP_SEC = 1800

# Session sample:
# 2 = giữ 2% session.
# Nếu vẫn đầy disk: đổi thành 1.
# Nếu muốn kỹ hơn và disk còn rộng: đổi thành 5.
SESSION_SAMPLE_MOD = 100
SESSION_SAMPLE_KEEP = 2

POSITIVE_EVENTS = {
    "view_phone",
    "contact_chat",
    "other_interaction",
    "contact_zalo",
    "contact_sms",
}

STRICT_CONTACT_EVENTS = {
    "view_phone",
    "contact_chat",
    "contact_zalo",
    "contact_sms",
}

EVENT_COLS = [
    "is_login",
    "session_id",
    "event_id",
    "item_id",
    "city_name",
    "category",
    "event_type",
    "event_ts",
    "surface",
    "device",
    "dwell_time_sec",
    "is_contact",
    "date",
]

print("EVENTS_PATH:", EVENTS_PATH)
print("Event files:", len(EVENT_FILES))
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SESSION SAMPLE:", f"{SESSION_SAMPLE_KEEP}/{SESSION_SAMPLE_MOD} = {SESSION_SAMPLE_KEEP}%")

if len(EVENT_FILES) == 0:
    raise FileNotFoundError(f"No parquet files found in {EVENTS_PATH}")

# ============================================================
# 1. HELPERS
# ============================================================

def clean_text_series(s, default="unknown"):
    return (
        s.astype("string")
         .fillna(default)
         .str.strip()
         .str.lower()
         .replace("", default)
    )

def normalize_login_group(s):
    x = clean_text_series(s)
    return np.where(x == "login", "login", "non_login")

def category_name_from_series(s):
    s_num = pd.to_numeric(s, errors="coerce").astype("Int64")
    mp = {
        1010: "1010_room_rental",
        1020: "1020_apartment",
        1030: "1030_house",
        1040: "1040_land_commercial",
        1050: "1050_new_project",
    }
    out = s_num.map(mp)
    return out.fillna("unknown_" + s_num.astype(str).fillna("NULL"))

def session_sample_mask(session_series, mod=100, keep=2):
    h = pd.util.hash_pandas_object(session_series.astype("string"), index=False).values
    return (h % mod) < keep

def write_parquet(df, path):
    df.to_parquet(path, index=False, compression="zstd")

def save_df(df, name, show_head=30):
    csv_path = os.path.join(TABLE_DIR, f"{name}.csv")
    parquet_path = os.path.join(TABLE_DIR, f"{name}.parquet")

    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    try:
        df.to_parquet(parquet_path, index=False, compression="zstd")
    except Exception as e:
        print("[WARN] Cannot save parquet:", e)

    print("[SAVE TABLE]", csv_path)
    display(df.head(show_head))
    return df

def plotly_safe_df(df):
    return df.copy().astype(object).where(pd.notna(df), None)

chart_index = []

def save_fig(fig, name, title=None):
    html_path = os.path.join(FIG_DIR, f"{name}.html")
    fig.write_html(html_path)
    print("[SAVE FIG]", html_path)

    chart_index.append({
        "name": name,
        "title": title if title else name,
        "html_path": html_path,
    })

    fig.show()

def run_duck(query, name=None, show_head=30, memory_limit="6GB"):
    con = duckdb.connect()
    con.execute("PRAGMA threads=2")
    con.execute(f"PRAGMA memory_limit='{memory_limit}'")
    con.execute("PRAGMA preserve_insertion_order=false")
    df = con.execute(query).df()
    con.close()

    if name:
        return save_df(df, name, show_head=show_head)
    return df

def make_label_event_category(df):
    df = df.copy()
    df["label"] = (
        df["event_type_clean"].astype(str)
        + " | "
        + df["category_name"].astype(str)
    )
    return df

def safe_rate(num, den):
    return num / den.replace(0, np.nan)

# ============================================================
# 2. MAP RAW EVENTS -> SAMPLED SESSION PARTIALS
# ============================================================

part_id = 0

for file_idx, fp in enumerate(EVENT_FILES):
    print(f"\n[SCAN] file {file_idx + 1}/{len(EVENT_FILES)}: {os.path.basename(fp)}")

    pf = pq.ParquetFile(fp)

    for batch_idx, batch in enumerate(pf.iter_batches(batch_size=BATCH_SIZE, columns=EVENT_COLS)):
        df = batch.to_pandas()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        df = df[df["session_id"].notna()].copy()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        df["session_id"] = df["session_id"].astype("string")

        # Sample theo session_id. Vì hash deterministic nên cùng session luôn được giữ/bỏ nhất quán.
        df = df[
            session_sample_mask(
                df["session_id"],
                mod=SESSION_SAMPLE_MOD,
                keep=SESSION_SAMPLE_KEEP
            )
        ].copy()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        # -----------------------------
        # Clean fields
        # -----------------------------
        df["is_login_group"] = normalize_login_group(df["is_login"])
        df["event_type_clean"] = clean_text_series(df["event_type"])
        df["city_clean"] = clean_text_series(df["city_name"])
        df["surface_clean"] = clean_text_series(df["surface"])
        df["device_clean"] = clean_text_series(df["device"])

        df["category"] = pd.to_numeric(df["category"], errors="coerce").astype("Int64")
        df["category_name"] = category_name_from_series(df["category"])

        df["active_date"] = pd.to_datetime(df["date"], errors="coerce").dt.floor("D")
        df["event_ts_clean"] = pd.to_datetime(df["event_ts"], errors="coerce")

        df["hour"] = df["event_ts_clean"].dt.hour.fillna(-1).astype("int16")
        df["dow_num"] = df["active_date"].dt.dayofweek.fillna(-1).astype("int16")
        df["dow_name"] = df["active_date"].dt.day_name().fillna("unknown")

        df["is_contact_int"] = pd.to_numeric(df["is_contact"], errors="coerce").fillna(0).astype("int8")
        df["is_positive_int"] = df["event_type_clean"].isin(POSITIVE_EVENTS).astype("int8")
        df["is_strict_contact_int"] = df["event_type_clean"].isin(STRICT_CONTACT_EVENTS).astype("int8")
        df["is_pageview_int"] = (df["event_type_clean"] == "pageview").astype("int8")

        dwell_raw = pd.to_numeric(df["dwell_time_sec"], errors="coerce").fillna(0).clip(lower=0)
        df["dwell_raw_sec"] = dwell_raw
        df["dwell_capped_sec"] = dwell_raw.clip(upper=DWELL_CAP_SEC)
        df["has_dwell"] = (dwell_raw > 0).astype("int8")

        # ====================================================
        # A. Session summary partial
        # ====================================================

        df["login_rows"] = (df["is_login_group"] == "login").astype("int8")
        df["non_login_rows"] = (df["is_login_group"] == "non_login").astype("int8")
        df["login_ts"] = df["event_ts_clean"].where(df["is_login_group"] == "login", pd.NaT)
        df["non_login_ts"] = df["event_ts_clean"].where(df["is_login_group"] == "non_login", pd.NaT)

        g_session = (
            df.groupby("session_id", observed=True)
            .agg(
                first_event_ts=("event_ts_clean", "min"),
                last_event_ts=("event_ts_clean", "max"),
                first_active_date=("active_date", "min"),
                last_active_date=("active_date", "max"),
                first_login_ts=("login_ts", "min"),
                first_non_login_ts=("non_login_ts", "min"),

                event_rows=("event_type_clean", "size"),
                login_event_rows=("login_rows", "sum"),
                non_login_event_rows=("non_login_rows", "sum"),

                pageview_rows=("is_pageview_int", "sum"),
                positive_event_rows=("is_positive_int", "sum"),
                strict_contact_event_rows=("is_strict_contact_int", "sum"),
                contact_flag_rows=("is_contact_int", "sum"),

                dwell_raw_sum_sec=("dwell_raw_sec", "sum"),
                dwell_capped_sum_sec=("dwell_capped_sec", "sum"),
                dwell_event_count=("has_dwell", "sum"),
            )
            .reset_index()
        )

        write_parquet(
            g_session,
            os.path.join(PARTIAL_SESSION_DIR, f"part_{part_id:07d}.parquet")
        )

        # ====================================================
        # B. session_id × event_type × category
        # ====================================================

        g_event_cat = (
            df.groupby(
                ["session_id", "event_type_clean", "category", "category_name"],
                observed=True,
                dropna=False,
            )
            .agg(
                rows=("event_type_clean", "size"),
                positive_event_rows=("is_positive_int", "sum"),
                strict_contact_event_rows=("is_strict_contact_int", "sum"),
                contact_flag_rows=("is_contact_int", "sum"),
                dwell_capped_sum_sec=("dwell_capped_sec", "sum"),
                dwell_event_count=("has_dwell", "sum"),
            )
            .reset_index()
        )

        write_parquet(
            g_event_cat,
            os.path.join(PARTIAL_EVENT_CAT_DIR, f"part_{part_id:07d}.parquet")
        )

        g_unique = df[
            ["session_id", "event_type_clean", "category", "category_name"]
        ].drop_duplicates()

        write_parquet(
            g_unique,
            os.path.join(PARTIAL_UNIQUE_EVENT_CAT_DIR, f"part_{part_id:07d}.parquet")
        )

        # ====================================================
        # C. session_id × city × category
        # ====================================================

        g_city_cat = (
            df.groupby(
                ["session_id", "city_clean", "category", "category_name"],
                observed=True,
                dropna=False,
            )
            .agg(
                rows=("event_type_clean", "size"),
                pageview_rows=("is_pageview_int", "sum"),
                positive_event_rows=("is_positive_int", "sum"),
                strict_contact_event_rows=("is_strict_contact_int", "sum"),
                contact_flag_rows=("is_contact_int", "sum"),
                dwell_capped_sum_sec=("dwell_capped_sec", "sum"),
                dwell_event_count=("has_dwell", "sum"),
            )
            .reset_index()
        )

        write_parquet(
            g_city_cat,
            os.path.join(PARTIAL_CITY_CAT_DIR, f"part_{part_id:07d}.parquet")
        )

        # ====================================================
        # D. strict contact × city × category
        # ====================================================

        contact_df = df[df["event_type_clean"].isin(STRICT_CONTACT_EVENTS)].copy()

        if len(contact_df) > 0:
            g_contact_city_cat = (
                contact_df.groupby(
                    ["session_id", "event_type_clean", "city_clean", "category", "category_name"],
                    observed=True,
                    dropna=False,
                )
                .agg(
                    rows=("event_type_clean", "size"),
                    dwell_capped_sum_sec=("dwell_capped_sec", "sum"),
                    dwell_event_count=("has_dwell", "sum"),
                )
                .reset_index()
            )

            write_parquet(
                g_contact_city_cat,
                os.path.join(PARTIAL_CONTACT_CITY_CAT_DIR, f"part_{part_id:07d}.parquet")
            )

        # ====================================================
        # E. session_id × date/hour
        # ====================================================

        time_df = df[df["active_date"].notna()].copy()

        if len(time_df) > 0:
            g_hour = (
                time_df.groupby(
                    ["session_id", "active_date", "hour", "dow_num", "dow_name"],
                    observed=True,
                    dropna=False,
                )
                .agg(
                    rows=("event_type_clean", "size"),
                    pageview_rows=("is_pageview_int", "sum"),
                    positive_event_rows=("is_positive_int", "sum"),
                    strict_contact_event_rows=("is_strict_contact_int", "sum"),
                    contact_flag_rows=("is_contact_int", "sum"),
                    dwell_capped_sum_sec=("dwell_capped_sec", "sum"),
                    dwell_event_count=("has_dwell", "sum"),
                )
                .reset_index()
            )

            write_parquet(
                g_hour,
                os.path.join(PARTIAL_HOUR_DIR, f"part_{part_id:07d}.parquet")
            )

        # ====================================================
        # F. session_id × device × surface
        # ====================================================

        g_device_surface = (
            df.groupby(
                ["session_id", "device_clean", "surface_clean"],
                observed=True,
                dropna=False,
            )
            .agg(
                rows=("event_type_clean", "size"),
                pageview_rows=("is_pageview_int", "sum"),
                positive_event_rows=("is_positive_int", "sum"),
                strict_contact_event_rows=("is_strict_contact_int", "sum"),
                contact_flag_rows=("is_contact_int", "sum"),
            )
            .reset_index()
        )

        write_parquet(
            g_device_surface,
            os.path.join(PARTIAL_DEVICE_SURFACE_DIR, f"part_{part_id:07d}.parquet")
        )

        part_id += 1

        del df
        gc.collect()

print("\nDONE MAP.")
print("Partial parts:", part_id)

# ============================================================
# 3. REDUCE SESSION SUMMARY
# ============================================================

SESSION_PARTIAL_GLOB = os.path.join(PARTIAL_SESSION_DIR, "*.parquet").replace("\\", "/")
SESSION_SUMMARY_PATH = os.path.join(CACHE_DIR, "session_summary_sample.parquet").replace("\\", "/")

con = duckdb.connect()
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='6GB'")
con.execute("PRAGMA preserve_insertion_order=false")

con.execute(f"""
COPY (
    WITH sess AS (
        SELECT
            CAST(session_id AS VARCHAR) AS session_id,

            MIN(first_event_ts) AS first_event_ts,
            MAX(last_event_ts) AS last_event_ts,
            MIN(first_active_date) AS first_active_date,
            MAX(last_active_date) AS last_active_date,
            MIN(first_login_ts) AS first_login_ts,
            MIN(first_non_login_ts) AS first_non_login_ts,

            SUM(event_rows) AS event_rows,
            SUM(login_event_rows) AS login_event_rows,
            SUM(non_login_event_rows) AS non_login_event_rows,

            SUM(pageview_rows) AS pageview_rows,
            SUM(positive_event_rows) AS positive_event_rows,
            SUM(strict_contact_event_rows) AS strict_contact_event_rows,
            SUM(contact_flag_rows) AS contact_flag_rows,

            SUM(dwell_raw_sum_sec) AS dwell_raw_sum_sec,
            SUM(dwell_capped_sum_sec) AS dwell_capped_sum_sec,
            SUM(dwell_event_count) AS dwell_event_count
        FROM read_parquet('{SESSION_PARTIAL_GLOB}')
        GROUP BY session_id
    ),

    scored AS (
        SELECT
            *,

            CASE
                WHEN login_event_rows > 0 AND non_login_event_rows = 0 THEN 'login_only'
                WHEN non_login_event_rows > 0 AND login_event_rows = 0 THEN 'non_login_only'
                WHEN login_event_rows > 0
                 AND non_login_event_rows > 0
                 AND first_non_login_ts <= first_login_ts
                    THEN 'nonlogin_then_login_same_session'
                WHEN login_event_rows > 0
                 AND non_login_event_rows > 0
                 AND first_login_ts < first_non_login_ts
                    THEN 'login_then_nonlogin_same_session'
                ELSE 'unknown'
            END AS session_type,

            DATE_DIFF('second', first_event_ts, last_event_ts) AS session_duration_sec,

            dwell_capped_sum_sec * 1.0 / NULLIF(dwell_event_count, 0) AS avg_dwell_per_dwell_event_sec,
            positive_event_rows * 1.0 / NULLIF(event_rows, 0) AS positive_event_rate_in_session,
            strict_contact_event_rows * 1.0 / NULLIF(event_rows, 0) AS strict_contact_event_rate_in_session,
            contact_flag_rows * 1.0 / NULLIF(event_rows, 0) AS contact_flag_rate_in_session,

            EXTRACT(hour FROM first_event_ts) AS first_hour,
            CAST(first_active_date AS DATE) AS session_start_date
        FROM sess
    )

    SELECT *
    FROM scored
) TO '{SESSION_SUMMARY_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.close()

print("[SAVE]", SESSION_SUMMARY_PATH)

# ============================================================
# 4. REDUCE DETAIL TABLES BY JOINING session_type
# ============================================================

EVENT_CAT_GLOB = os.path.join(PARTIAL_EVENT_CAT_DIR, "*.parquet").replace("\\", "/")
UNIQUE_EVENT_CAT_GLOB = os.path.join(PARTIAL_UNIQUE_EVENT_CAT_DIR, "*.parquet").replace("\\", "/")
CITY_CAT_GLOB = os.path.join(PARTIAL_CITY_CAT_DIR, "*.parquet").replace("\\", "/")
CONTACT_CITY_CAT_GLOB = os.path.join(PARTIAL_CONTACT_CITY_CAT_DIR, "*.parquet").replace("\\", "/")
HOUR_GLOB = os.path.join(PARTIAL_HOUR_DIR, "*.parquet").replace("\\", "/")
DEVICE_SURFACE_GLOB = os.path.join(PARTIAL_DEVICE_SURFACE_DIR, "*.parquet").replace("\\", "/")

# ------------------------------------------------------------
# 4.1 Session type summary
# ------------------------------------------------------------

session_type_summary = run_duck(f"""
SELECT
    session_type,

    COUNT(*) AS sampled_sessions,

    SUM(event_rows) AS event_rows,
    AVG(event_rows) AS avg_event_rows_per_session,
    QUANTILE_CONT(event_rows, 0.50) AS median_event_rows_per_session,
    QUANTILE_CONT(event_rows, 0.90) AS p90_event_rows_per_session,

    AVG(pageview_rows) AS avg_pageview_rows_per_session,
    AVG(positive_event_rows) AS avg_positive_event_rows_per_session,
    AVG(strict_contact_event_rows) AS avg_strict_contact_event_rows_per_session,
    AVG(contact_flag_rows) AS avg_contact_flag_rows_per_session,

    SUM(CASE WHEN positive_event_rows > 0 THEN 1 ELSE 0 END) AS sessions_with_positive,
    SUM(CASE WHEN strict_contact_event_rows > 0 THEN 1 ELSE 0 END) AS sessions_with_strict_contact,
    SUM(CASE WHEN contact_flag_rows > 0 THEN 1 ELSE 0 END) AS sessions_with_contact_flag,

    SUM(CASE WHEN positive_event_rows > 0 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0) AS positive_session_rate,
    SUM(CASE WHEN strict_contact_event_rows > 0 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0) AS strict_contact_session_rate,
    SUM(CASE WHEN contact_flag_rows > 0 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0) AS contact_flag_session_rate,

    AVG(dwell_capped_sum_sec) AS avg_total_dwell_capped_sec_per_session,
    QUANTILE_CONT(dwell_capped_sum_sec, 0.50) AS median_total_dwell_capped_sec_per_session,
    QUANTILE_CONT(dwell_capped_sum_sec, 0.75) AS p75_total_dwell_capped_sec_per_session,
    QUANTILE_CONT(dwell_capped_sum_sec, 0.90) AS p90_total_dwell_capped_sec_per_session,
    QUANTILE_CONT(dwell_capped_sum_sec, 0.95) AS p95_total_dwell_capped_sec_per_session,

    AVG(session_duration_sec) AS avg_session_duration_sec,
    QUANTILE_CONT(session_duration_sec, 0.50) AS median_session_duration_sec,
    QUANTILE_CONT(session_duration_sec, 0.90) AS p90_session_duration_sec
FROM read_parquet('{SESSION_SUMMARY_PATH}')
GROUP BY session_type
ORDER BY sampled_sessions DESC
""", "00_session_type_summary_sample", show_head=50)

# ------------------------------------------------------------
# 4.2 session_type × event_type × category
# ------------------------------------------------------------

event_cat_rows = run_duck(f"""
WITH ec AS (
    SELECT
        CAST(session_id AS VARCHAR) AS session_id,
        event_type_clean,
        category,
        category_name,
        SUM(rows) AS rows,
        SUM(positive_event_rows) AS positive_event_rows,
        SUM(strict_contact_event_rows) AS strict_contact_event_rows,
        SUM(contact_flag_rows) AS contact_flag_rows,
        SUM(dwell_capped_sum_sec) AS dwell_capped_sum_sec,
        SUM(dwell_event_count) AS dwell_event_count
    FROM read_parquet('{EVENT_CAT_GLOB}')
    GROUP BY session_id, event_type_clean, category, category_name
),

sess AS (
    SELECT
        session_id,
        session_type
    FROM read_parquet('{SESSION_SUMMARY_PATH}')
)

SELECT
    s.session_type,
    e.event_type_clean,
    e.category,
    e.category_name,

    SUM(e.rows) AS rows,
    SUM(e.positive_event_rows) AS positive_event_rows,
    SUM(e.strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(e.contact_flag_rows) AS contact_flag_rows,
    SUM(e.dwell_capped_sum_sec) AS dwell_capped_sum_sec,
    SUM(e.dwell_event_count) AS dwell_event_count,

    SUM(e.positive_event_rows) * 1.0 / NULLIF(SUM(e.rows), 0) AS positive_event_rate,
    SUM(e.strict_contact_event_rows) * 1.0 / NULLIF(SUM(e.rows), 0) AS strict_contact_event_rate,
    SUM(e.contact_flag_rows) * 1.0 / NULLIF(SUM(e.rows), 0) AS contact_flag_rate,
    SUM(e.dwell_capped_sum_sec) * 1.0 / NULLIF(SUM(e.dwell_event_count), 0) AS avg_dwell_per_dwell_event_sec
FROM ec e
INNER JOIN sess s
    ON e.session_id = s.session_id
GROUP BY s.session_type, e.event_type_clean, e.category, e.category_name
ORDER BY rows DESC
""")

event_cat_sessions = run_duck(f"""
WITH u AS (
    SELECT DISTINCT
        CAST(session_id AS VARCHAR) AS session_id,
        event_type_clean,
        category,
        category_name
    FROM read_parquet('{UNIQUE_EVENT_CAT_GLOB}')
),

sess AS (
    SELECT
        session_id,
        session_type
    FROM read_parquet('{SESSION_SUMMARY_PATH}')
)

SELECT
    s.session_type,
    u.event_type_clean,
    u.category,
    u.category_name,
    COUNT(DISTINCT u.session_id) AS sampled_sessions_with_event
FROM u
INNER JOIN sess s
    ON u.session_id = s.session_id
GROUP BY s.session_type, u.event_type_clean, u.category, u.category_name
""")

session_type_event_category = event_cat_rows.merge(
    event_cat_sessions,
    on=["session_type", "event_type_clean", "category", "category_name"],
    how="left"
)

session_type_event_category["row_share_inside_session_type"] = (
    session_type_event_category["rows"]
    / session_type_event_category.groupby("session_type")["rows"].transform("sum")
)

session_type_event_category["row_share_inside_session_type_pct"] = (
    session_type_event_category["row_share_inside_session_type"] * 100
)

session_type_event_category = session_type_event_category.sort_values("rows", ascending=False)

save_df(
    session_type_event_category,
    "01_session_type_event_category_detail",
    show_head=100
)

# ------------------------------------------------------------
# 4.3 session_type × city × category
# ------------------------------------------------------------

session_type_city_category = run_duck(f"""
WITH cc AS (
    SELECT
        CAST(session_id AS VARCHAR) AS session_id,
        city_clean,
        category,
        category_name,
        SUM(rows) AS rows,
        SUM(pageview_rows) AS pageview_rows,
        SUM(positive_event_rows) AS positive_event_rows,
        SUM(strict_contact_event_rows) AS strict_contact_event_rows,
        SUM(contact_flag_rows) AS contact_flag_rows,
        SUM(dwell_capped_sum_sec) AS dwell_capped_sum_sec,
        SUM(dwell_event_count) AS dwell_event_count
    FROM read_parquet('{CITY_CAT_GLOB}')
    GROUP BY session_id, city_clean, category, category_name
),

sess AS (
    SELECT
        session_id,
        session_type
    FROM read_parquet('{SESSION_SUMMARY_PATH}')
)

SELECT
    s.session_type,
    c.city_clean,
    c.category,
    c.category_name,

    SUM(c.rows) AS rows,
    SUM(c.pageview_rows) AS pageview_rows,
    SUM(c.positive_event_rows) AS positive_event_rows,
    SUM(c.strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(c.contact_flag_rows) AS contact_flag_rows,
    SUM(c.dwell_capped_sum_sec) AS dwell_capped_sum_sec,
    SUM(c.dwell_event_count) AS dwell_event_count,

    SUM(c.positive_event_rows) * 1.0 / NULLIF(SUM(c.rows), 0) AS positive_event_rate,
    SUM(c.strict_contact_event_rows) * 1.0 / NULLIF(SUM(c.rows), 0) AS strict_contact_event_rate,
    SUM(c.contact_flag_rows) * 1.0 / NULLIF(SUM(c.rows), 0) AS contact_flag_rate,
    SUM(c.dwell_capped_sum_sec) * 1.0 / NULLIF(SUM(c.dwell_event_count), 0) AS avg_dwell_per_dwell_event_sec
FROM cc c
INNER JOIN sess s
    ON c.session_id = s.session_id
GROUP BY s.session_type, c.city_clean, c.category, c.category_name
ORDER BY rows DESC
""", "02_session_type_city_category_detail", show_head=100)

# ------------------------------------------------------------
# 4.4 session_type × contact event × city × category
# ------------------------------------------------------------

if len(glob.glob(CONTACT_CITY_CAT_GLOB)) > 0:
    session_type_contact_city_category = run_duck(f"""
    WITH ccc AS (
        SELECT
            CAST(session_id AS VARCHAR) AS session_id,
            event_type_clean,
            city_clean,
            category,
            category_name,
            SUM(rows) AS rows,
            SUM(dwell_capped_sum_sec) AS dwell_capped_sum_sec,
            SUM(dwell_event_count) AS dwell_event_count
        FROM read_parquet('{CONTACT_CITY_CAT_GLOB}')
        GROUP BY session_id, event_type_clean, city_clean, category, category_name
    ),

    sess AS (
        SELECT
            session_id,
            session_type
        FROM read_parquet('{SESSION_SUMMARY_PATH}')
    )

    SELECT
        s.session_type,
        c.event_type_clean,
        c.city_clean,
        c.category,
        c.category_name,

        SUM(c.rows) AS rows,
        SUM(c.dwell_capped_sum_sec) AS dwell_capped_sum_sec,
        SUM(c.dwell_event_count) AS dwell_event_count,
        SUM(c.dwell_capped_sum_sec) * 1.0 / NULLIF(SUM(c.dwell_event_count), 0) AS avg_dwell_per_dwell_event_sec
    FROM ccc c
    INNER JOIN sess s
        ON c.session_id = s.session_id
    GROUP BY s.session_type, c.event_type_clean, c.city_clean, c.category, c.category_name
    ORDER BY rows DESC
    """, "03_session_type_contact_city_category_detail", show_head=100)
else:
    session_type_contact_city_category = pd.DataFrame()
    print("[SKIP] No contact partial files.")

# ------------------------------------------------------------
# 4.5 session_type × hour
# ------------------------------------------------------------

session_type_hour = run_duck(f"""
WITH h AS (
    SELECT
        CAST(session_id AS VARCHAR) AS session_id,
        active_date,
        hour,
        dow_num,
        dow_name,
        SUM(rows) AS rows,
        SUM(pageview_rows) AS pageview_rows,
        SUM(positive_event_rows) AS positive_event_rows,
        SUM(strict_contact_event_rows) AS strict_contact_event_rows,
        SUM(contact_flag_rows) AS contact_flag_rows,
        SUM(dwell_capped_sum_sec) AS dwell_capped_sum_sec,
        SUM(dwell_event_count) AS dwell_event_count
    FROM read_parquet('{HOUR_GLOB}')
    WHERE hour >= 0
    GROUP BY session_id, active_date, hour, dow_num, dow_name
),

sess AS (
    SELECT
        session_id,
        session_type
    FROM read_parquet('{SESSION_SUMMARY_PATH}')
)

SELECT
    h.active_date,
    h.hour,
    h.dow_num,
    h.dow_name,
    s.session_type,

    SUM(h.rows) AS rows,
    SUM(h.pageview_rows) AS pageview_rows,
    SUM(h.positive_event_rows) AS positive_event_rows,
    SUM(h.strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(h.contact_flag_rows) AS contact_flag_rows,
    SUM(h.dwell_capped_sum_sec) AS dwell_capped_sum_sec,
    SUM(h.dwell_event_count) AS dwell_event_count,

    SUM(h.positive_event_rows) * 1.0 / NULLIF(SUM(h.rows), 0) AS positive_event_rate,
    SUM(h.strict_contact_event_rows) * 1.0 / NULLIF(SUM(h.rows), 0) AS strict_contact_event_rate,
    SUM(h.dwell_capped_sum_sec) * 1.0 / NULLIF(SUM(h.dwell_event_count), 0) AS avg_dwell_per_dwell_event_sec
FROM h
INNER JOIN sess s
    ON h.session_id = s.session_id
GROUP BY h.active_date, h.hour, h.dow_num, h.dow_name, s.session_type
ORDER BY h.active_date, h.hour, s.session_type
""", "04_session_type_hourly_detail", show_head=100)

# ------------------------------------------------------------
# 4.6 session_type × device × surface
# ------------------------------------------------------------

session_type_device_surface = run_duck(f"""
WITH ds AS (
    SELECT
        CAST(session_id AS VARCHAR) AS session_id,
        device_clean,
        surface_clean,
        SUM(rows) AS rows,
        SUM(pageview_rows) AS pageview_rows,
        SUM(positive_event_rows) AS positive_event_rows,
        SUM(strict_contact_event_rows) AS strict_contact_event_rows,
        SUM(contact_flag_rows) AS contact_flag_rows
    FROM read_parquet('{DEVICE_SURFACE_GLOB}')
    GROUP BY session_id, device_clean, surface_clean
),

sess AS (
    SELECT
        session_id,
        session_type
    FROM read_parquet('{SESSION_SUMMARY_PATH}')
)

SELECT
    s.session_type,
    d.device_clean,
    d.surface_clean,

    SUM(d.rows) AS rows,
    SUM(d.pageview_rows) AS pageview_rows,
    SUM(d.positive_event_rows) AS positive_event_rows,
    SUM(d.strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(d.contact_flag_rows) AS contact_flag_rows,

    SUM(d.positive_event_rows) * 1.0 / NULLIF(SUM(d.rows), 0) AS positive_event_rate,
    SUM(d.strict_contact_event_rows) * 1.0 / NULLIF(SUM(d.rows), 0) AS strict_contact_event_rate,
    SUM(d.contact_flag_rows) * 1.0 / NULLIF(SUM(d.rows), 0) AS contact_flag_rate
FROM ds d
INNER JOIN sess s
    ON d.session_id = s.session_id
GROUP BY s.session_type, d.device_clean, d.surface_clean
ORDER BY rows DESC
""", "05_session_type_device_surface_detail", show_head=100)

# ============================================================
# 5. VISUALIZATION
# ============================================================

focus_types = [
    "non_login_only",
    "nonlogin_then_login_same_session",
    "login_only",
]

# ------------------------------------------------------------
# 5.1 Session type summary table
# ------------------------------------------------------------

df = session_type_summary.copy()

for c in ["positive_session_rate", "strict_contact_session_rate", "contact_flag_session_rate"]:
    if c in df.columns:
        df[c + "_pct"] = df[c] * 100

fig = go.Figure(data=[go.Table(
    header=dict(values=list(df.columns), align="left"),
    cells=dict(values=[df[c] for c in df.columns], align="left"),
)])
fig.update_layout(
    title=f"Session type summary — {SESSION_SAMPLE_KEEP}% sample",
    height=520,
)
save_fig(fig, "00_session_type_summary_table")

# ------------------------------------------------------------
# 5.2 Dwell by session type
# ------------------------------------------------------------

fig = px.bar(
    plotly_safe_df(df),
    x="session_type",
    y=[
        "avg_total_dwell_capped_sec_per_session",
        "median_total_dwell_capped_sec_per_session",
        "p90_total_dwell_capped_sec_per_session",
    ],
    barmode="group",
    hover_data=[
        "sampled_sessions",
        "avg_event_rows_per_session",
        "median_event_rows_per_session",
        "positive_session_rate_pct",
        "strict_contact_session_rate_pct",
    ],
    title=f"Dwell time by session type — {SESSION_SAMPLE_KEEP}% sample",
)
fig.update_layout(
    xaxis_title="Session type",
    yaxis_title="Dwell time, capped seconds",
    xaxis_tickangle=-20,
    height=620,
)
save_fig(fig, "01_dwell_time_by_session_type")

# ------------------------------------------------------------
# 5.3 Interaction intensity by session type
# ------------------------------------------------------------

fig = px.bar(
    plotly_safe_df(df),
    x="session_type",
    y=[
        "avg_event_rows_per_session",
        "avg_pageview_rows_per_session",
        "avg_positive_event_rows_per_session",
        "avg_strict_contact_event_rows_per_session",
    ],
    barmode="group",
    hover_data=[
        "sampled_sessions",
        "median_event_rows_per_session",
        "p90_event_rows_per_session",
        "median_total_dwell_capped_sec_per_session",
    ],
    title=f"Interaction intensity by session type — {SESSION_SAMPLE_KEEP}% sample",
)
fig.update_layout(
    xaxis_title="Session type",
    yaxis_title="Average rows per session",
    xaxis_tickangle=-20,
    height=620,
)
save_fig(fig, "02_interaction_intensity_by_session_type")

# ------------------------------------------------------------
# 5.4 Event-category by session type
# ------------------------------------------------------------

df_ec = make_label_event_category(session_type_event_category)

plot_df = (
    df_ec[df_ec["session_type"].isin(focus_types)]
    .sort_values("rows", ascending=False)
    .groupby("session_type", group_keys=False)
    .head(15)
)

fig = px.bar(
    plotly_safe_df(plot_df),
    x="label",
    y="rows",
    color="session_type",
    barmode="group",
    hover_data=[
        "sampled_sessions_with_event",
        "row_share_inside_session_type_pct",
        "positive_event_rows",
        "strict_contact_event_rows",
        "contact_flag_rows",
        "positive_event_rate",
        "strict_contact_event_rate",
        "avg_dwell_per_dwell_event_sec",
    ],
    title="What each session type does: event type × category",
)
fig.update_layout(
    xaxis_title="event_type | category",
    yaxis_title="Rows in sampled sessions",
    xaxis_tickangle=-35,
    height=800,
)
save_fig(fig, "03_session_type_event_category_comparison")

# ------------------------------------------------------------
# 5.5 Converted sessions only
# ------------------------------------------------------------

converted = df_ec[df_ec["session_type"] == "nonlogin_then_login_same_session"].copy()
plot_df = converted.sort_values("rows", ascending=False).head(TOP_N)

fig = px.bar(
    plotly_safe_df(plot_df),
    x="label",
    y="rows",
    color="event_type_clean",
    hover_data=[
        "sampled_sessions_with_event",
        "row_share_inside_session_type_pct",
        "positive_event_rows",
        "strict_contact_event_rows",
        "contact_flag_rows",
        "positive_event_rate",
        "strict_contact_event_rate",
        "avg_dwell_per_dwell_event_sec",
    ],
    title="Converted sessions only: non-login → login, event type × category",
)
fig.update_layout(
    xaxis_title="event_type | category",
    yaxis_title="Rows in converted sampled sessions",
    xaxis_tickangle=-35,
    height=720,
)
save_fig(fig, "04_converted_sessions_event_category")

# ------------------------------------------------------------
# 5.6 Strict contact by session type
# ------------------------------------------------------------

contact_plot = df_ec[
    df_ec["event_type_clean"].isin(STRICT_CONTACT_EVENTS)
    & df_ec["session_type"].isin(focus_types)
].copy()

plot_df = (
    contact_plot.sort_values("rows", ascending=False)
    .groupby("session_type", group_keys=False)
    .head(12)
)

fig = px.bar(
    plotly_safe_df(plot_df),
    x="label",
    y="rows",
    color="session_type",
    barmode="group",
    hover_data=[
        "sampled_sessions_with_event",
        "row_share_inside_session_type_pct",
        "strict_contact_event_rate",
        "avg_dwell_per_dwell_event_sec",
    ],
    title="Strict contact events by session type × category",
)
fig.update_layout(
    xaxis_title="contact event | category",
    yaxis_title="Rows",
    xaxis_tickangle=-35,
    height=720,
)
save_fig(fig, "05_contact_event_category_by_session_type")

# ------------------------------------------------------------
# 5.7 Event-category overrepresented in converted sessions
# ------------------------------------------------------------

share_pivot = (
    df_ec[df_ec["session_type"].isin(["non_login_only", "nonlogin_then_login_same_session"])]
    .pivot_table(
        index=["event_type_clean", "category", "category_name", "label"],
        columns="session_type",
        values="row_share_inside_session_type",
        aggfunc="sum",
    )
    .fillna(0)
    .reset_index()
)

if "non_login_only" not in share_pivot.columns:
    share_pivot["non_login_only"] = 0

if "nonlogin_then_login_same_session" not in share_pivot.columns:
    share_pivot["nonlogin_then_login_same_session"] = 0

share_pivot["converted_minus_nonlogin_only_share"] = (
    share_pivot["nonlogin_then_login_same_session"] - share_pivot["non_login_only"]
)

share_pivot["converted_minus_nonlogin_only_share_pct_point"] = (
    share_pivot["converted_minus_nonlogin_only_share"] * 100
)

share_pivot = share_pivot.sort_values(
    "converted_minus_nonlogin_only_share_pct_point",
    ascending=False
)

save_df(
    share_pivot,
    "06_converted_vs_nonlogin_only_event_share_lift",
    show_head=100
)

plot_df = share_pivot.head(TOP_N)

fig = px.bar(
    plotly_safe_df(plot_df),
    x="label",
    y="converted_minus_nonlogin_only_share_pct_point",
    color="event_type_clean",
    hover_data=[
        "non_login_only",
        "nonlogin_then_login_same_session",
    ],
    title="Event-category overrepresented in converted sessions vs non-login-only",
)
fig.update_layout(
    xaxis_title="event_type | category",
    yaxis_title="Share lift in converted sessions, percentage points",
    xaxis_tickangle=-35,
    height=720,
)
save_fig(fig, "06_converted_vs_nonlogin_event_share_lift")

# ------------------------------------------------------------
# 5.8 Converted sessions city-category
# ------------------------------------------------------------

city_df = session_type_city_category.copy()
city_df["label"] = city_df["city_clean"].astype(str) + " | " + city_df["category_name"].astype(str)

converted_city = city_df[city_df["session_type"] == "nonlogin_then_login_same_session"].copy()
plot_df = converted_city.sort_values("rows", ascending=False).head(TOP_N)

fig = px.bar(
    plotly_safe_df(plot_df),
    x="label",
    y="rows",
    hover_data=[
        "pageview_rows",
        "positive_event_rows",
        "strict_contact_event_rows",
        "contact_flag_rows",
        "positive_event_rate",
        "strict_contact_event_rate",
        "avg_dwell_per_dwell_event_sec",
    ],
    title="Converted sessions: top city × category",
)
fig.update_layout(
    xaxis_title="city | category",
    yaxis_title="Rows",
    xaxis_tickangle=-35,
    height=720,
)
save_fig(fig, "07_converted_sessions_city_category")

# ------------------------------------------------------------
# 5.9 Converted sessions city-category heatmap
# ------------------------------------------------------------

top_cities = (
    converted_city.groupby("city_clean", as_index=False)["rows"]
    .sum()
    .sort_values("rows", ascending=False)
    .head(15)["city_clean"]
    .tolist()
)

if len(top_cities) > 0:
    heat = (
        converted_city[converted_city["city_clean"].isin(top_cities)]
        .pivot_table(
            index="city_clean",
            columns="category_name",
            values="rows",
            aggfunc="sum",
        )
        .fillna(0)
    )

    fig = px.imshow(
        heat,
        aspect="auto",
        title="Converted sessions heatmap: city × category",
        labels=dict(x="Category", y="City", color="Rows"),
    )
    fig.update_layout(height=650)
    save_fig(fig, "08_converted_sessions_city_category_heatmap")

# ------------------------------------------------------------
# 5.10 Hour-of-day by session type
# ------------------------------------------------------------

hour_df = (
    session_type_hour
    .groupby(["session_type", "hour"], as_index=False)
    .agg(
        rows=("rows", "sum"),
        positive_event_rows=("positive_event_rows", "sum"),
        strict_contact_event_rows=("strict_contact_event_rows", "sum"),
    )
)

hour_df = hour_df[hour_df["session_type"].isin(focus_types)].copy()
hour_df["share_inside_session_type"] = (
    hour_df["rows"] / hour_df.groupby("session_type")["rows"].transform("sum")
)

fig = px.line(
    plotly_safe_df(hour_df),
    x="hour",
    y="share_inside_session_type",
    color="session_type",
    markers=True,
    hover_data=[
        "rows",
        "positive_event_rows",
        "strict_contact_event_rows",
    ],
    title="Hour-of-day distribution by session type",
)
fig.update_layout(
    xaxis_title="Hour",
    yaxis_title="Share of rows inside session type",
    height=620,
)
save_fig(fig, "09_hour_of_day_by_session_type")

# ------------------------------------------------------------
# 5.11 Converted session weekday-hour heatmap
# ------------------------------------------------------------

converted_hour = session_type_hour[
    session_type_hour["session_type"] == "nonlogin_then_login_same_session"
].copy()

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

if len(converted_hour) > 0:
    converted_hour["dow_name"] = pd.Categorical(
        converted_hour["dow_name"],
        categories=dow_order,
        ordered=True
    )

    heat = (
        converted_hour.pivot_table(
            index="dow_name",
            columns="hour",
            values="rows",
            aggfunc="sum",
        )
        .reindex(dow_order)
        .fillna(0)
    )

    fig = px.imshow(
        heat,
        aspect="auto",
        title="Converted sessions: weekday × hour heatmap",
        labels=dict(x="Hour", y="Day of week", color="Rows"),
    )
    fig.update_layout(height=560)
    save_fig(fig, "10_converted_sessions_weekday_hour_heatmap")

# ------------------------------------------------------------
# 5.12 Device-surface by session type
# ------------------------------------------------------------

ds = session_type_device_surface[
    session_type_device_surface["session_type"].isin(focus_types)
].copy()

ds["label"] = ds["device_clean"].astype(str) + " | " + ds["surface_clean"].astype(str)

plot_df = (
    ds.sort_values("rows", ascending=False)
    .groupby("session_type", group_keys=False)
    .head(12)
)

fig = px.bar(
    plotly_safe_df(plot_df),
    x="label",
    y="rows",
    color="session_type",
    barmode="group",
    hover_data=[
        "pageview_rows",
        "positive_event_rows",
        "strict_contact_event_rows",
        "positive_event_rate",
        "strict_contact_event_rate",
    ],
    title="Device × surface by session type",
)
fig.update_layout(
    xaxis_title="device | surface",
    yaxis_title="Rows",
    xaxis_tickangle=-35,
    height=720,
)
save_fig(fig, "11_device_surface_by_session_type")

# ------------------------------------------------------------
# 5.13 Converted sessions strict contact city-category
# ------------------------------------------------------------

if len(session_type_contact_city_category) > 0:
    ccc = session_type_contact_city_category[
        session_type_contact_city_category["session_type"] == "nonlogin_then_login_same_session"
    ].copy()

    ccc["label"] = (
        ccc["event_type_clean"].astype(str)
        + " | "
        + ccc["city_clean"].astype(str)
        + " | "
        + ccc["category_name"].astype(str)
    )

    plot_df = ccc.sort_values("rows", ascending=False).head(TOP_N)

    fig = px.bar(
        plotly_safe_df(plot_df),
        x="label",
        y="rows",
        color="event_type_clean",
        hover_data=[
            "avg_dwell_per_dwell_event_sec",
        ],
        title="Converted sessions: strict contact event × city × category",
    )
    fig.update_layout(
        xaxis_title="contact event | city | category",
        yaxis_title="Rows",
        xaxis_tickangle=-35,
        height=760,
    )
    save_fig(fig, "12_converted_contact_city_category")

# ============================================================
# 6. SAVE CHART INDEX
# ============================================================

chart_index_df = pd.DataFrame(chart_index)
chart_index_path = os.path.join(OUTPUT_DIR, "99_chart_index.csv")
chart_index_df.to_csv(chart_index_path, index=False, encoding="utf-8-sig")

print("\nDONE.")
print("Output folder:", OUTPUT_DIR)
print("Tables:", TABLE_DIR)
print("Figures:", FIG_DIR)
print("Chart index:", chart_index_path)
display(chart_index_df)

print("""
MAIN TABLES:
00_session_type_summary_sample.csv
01_session_type_event_category_detail.csv
02_session_type_city_category_detail.csv
03_session_type_contact_city_category_detail.csv
04_session_type_hourly_detail.csv
05_session_type_device_surface_detail.csv
06_converted_vs_nonlogin_only_event_share_lift.csv

MAIN CHARTS:
00_session_type_summary_table.html
01_dwell_time_by_session_type.html
02_interaction_intensity_by_session_type.html
03_session_type_event_category_comparison.html
04_converted_sessions_event_category.html
05_contact_event_category_by_session_type.html
06_converted_vs_nonlogin_event_share_lift.html
07_converted_sessions_city_category.html
08_converted_sessions_city_category_heatmap.html
09_hour_of_day_by_session_type.html
10_converted_sessions_weekday_hour_heatmap.html
11_device_surface_by_session_type.html
12_converted_contact_city_category.html
""")

EVENTS_PATH: /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2/fact_user_events
Event files: 500
OUTPUT_DIR: /kaggle/working/eda_skipped_session_detail_from_scratch
SESSION SAMPLE: 2/100 = 2%

[SCAN] file 1/500: datathon_fact_user_events-000000000000.parquet

[SCAN] file 2/500: datathon_fact_user_events-000000000001.parquet

[SCAN] file 3/500: datathon_fact_user_events-000000000002.parquet

[SCAN] file 4/500: datathon_fact_user_events-000000000003.parquet

[SCAN] file 5/500: datathon_fact_user_events-000000000004.parquet

[SCAN] file 6/500: datathon_fact_user_events-000000000005.parquet

[SCAN] file 7/500: datathon_fact_user_events-000000000006.parquet

[SCAN] file 8/500: datathon_fact_user_events-000000000007.parquet

[SCAN] file 9/500: datathon_fact_user_events-000000000008.parquet

[SCAN] file 10/500: datathon_fact_user_events-000000000009.parquet

[SCAN] file 11/500: datathon_fact_user_events-000000000010.parquet

[SCAN] file 12/500: datathon_fact_user_events-000000000011.par

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[SAVE] /kaggle/working/eda_skipped_session_detail_from_scratch/cache/session_summary_sample.parquet
[SAVE TABLE] /kaggle/working/eda_skipped_session_detail_from_scratch/tables/00_session_type_summary_sample.csv


,session_type,sampled_sessions,event_rows,avg_event_rows_per_session,median_event_rows_per_session,p90_event_rows_per_session,avg_pageview_rows_per_session,avg_positive_event_rows_per_session,avg_strict_contact_event_rows_per_session,avg_contact_flag_rows_per_session,...,strict_contact_session_rate,contact_flag_session_rate,avg_total_dwell_capped_sec_per_session,median_total_dwell_capped_sec_per_session,p75_total_dwell_capped_sec_per_session,p90_total_dwell_capped_sec_per_session,p95_total_dwell_capped_sec_per_session,avg_session_duration_sec,median_session_duration_sec,p90_session_duration_sec
0,login_only,70956,1383411.0,19.496744,8.0,49.0,9.592734,9.904011,0.766137,9.904011,...,0.222363,0.701421,15637.733398,7200.0,19685.25,42097.0,62263.00,1163.416173,251.0,2224.5
1,non_login_only,32876,585917.0,17.822028,7.0,43.0,4.807002,13.015026,0.403030,13.015026,...,0.119936,0.814850,7223.644908,3600.0,9000.00,18891.0,29168.75,772.319260,98.0,1330.5
2,nonlogin_then_login_same_session,19253,729780.0,37.904742,22.0,86.0,14.253675,23.651067,1.245157,23.651067,...,0.335376,0.951904,22334.643952,13822.0,29234.00,53980.2,73202.80,2455.919753,786.0,3151.8
3,login_then_nonlogin_same_session,12102,533032.0,44.044951,27.0,101.0,16.375723,27.669228,1.432656,27.669228,...,0.361676,0.955214,26319.416460,16200.0,34200.00,62659.3,85511.65,3681.398447,817.0,3280.9


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

[SAVE FIG] /kaggle/working/eda_skipped_session_detail_from_scratch/figures/11_device_surface_by_session_type.html


[SAVE FIG] /kaggle/working/eda_skipped_session_detail_from_scratch/figures/12_converted_contact_city_category.html



DONE.
Output folder: /kaggle/working/eda_skipped_session_detail_from_scratch
Tables: /kaggle/working/eda_skipped_session_detail_from_scratch/tables
Figures: /kaggle/working/eda_skipped_session_detail_from_scratch/figures
Chart index: /kaggle/working/eda_skipped_session_detail_from_scratch/99_chart_index.csv


,name,title,html_path
0,00_session_type_summary_table,00_session_type_summary_table,/kaggle/working/eda_skipped_session_detail_fro...
1,01_dwell_time_by_session_type,01_dwell_time_by_session_type,/kaggle/working/eda_skipped_session_detail_fro...
2,02_interaction_intensity_by_session_type,02_interaction_intensity_by_session_type,/kaggle/working/eda_skipped_session_detail_fro...
3,03_session_type_event_category_comparison,03_session_type_event_category_comparison,/kaggle/working/eda_skipped_session_detail_fro...
4,04_converted_sessions_event_category,04_converted_sessions_event_category,/kaggle/working/eda_skipped_session_detail_fro...
5,05_contact_event_category_by_session_type,05_contact_event_category_by_session_type,/kaggle/working/eda_skipped_session_detail_fro...
6,06_converted_vs_nonlogin_event_share_lift,06_converted_vs_nonlogin_event_share_lift,/kaggle/working/eda_skipped_session_detail_fro...
7,07_converted_sessions_city_category,07_converted_sessions_city_category,/kaggle/working/eda_skipped_session_detail_fro...
8,08_converted_sessions_city_category_heatmap,08_converted_sessions_city_category_heatmap,/kaggle/working/eda_skipped_session_detail_fro...
9,09_hour_of_day_by_session_type,09_hour_of_day_by_session_type,/kaggle/working/eda_skipped_session_detail_fro...



MAIN TABLES:
00_session_type_summary_sample.csv
01_session_type_event_category_detail.csv
02_session_type_city_category_detail.csv
03_session_type_contact_city_category_detail.csv
04_session_type_hourly_detail.csv
05_session_type_device_surface_detail.csv
06_converted_vs_nonlogin_only_event_share_lift.csv

MAIN CHARTS:
00_session_type_summary_table.html
01_dwell_time_by_session_type.html
02_interaction_intensity_by_session_type.html
03_session_type_event_category_comparison.html
04_converted_sessions_event_category.html
05_contact_event_category_by_session_type.html
06_converted_vs_nonlogin_event_share_lift.html
07_converted_sessions_city_category.html
08_converted_sessions_city_category_heatmap.html
09_hour_of_day_by_session_type.html
10_converted_sessions_weekday_hour_heatmap.html
11_device_surface_by_session_type.html
12_converted_contact_city_category.html

